In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/prince7489/e-commerce-sales/Ecommerce_Sales_Data_2024_2025.csv


# E commerce India (AI model training and Data Analytics Dashboard)

Today we are going to look into an E commerce dataset to see what are the practical uses of it. and to warm up our data analytics and AI training. see the potential business gain analysis in the data. eg like Profitability and Discount Anal

In [2]:
df = pd.read_csv('/kaggle/input/datasets/prince7489/e-commerce-sales/Ecommerce_Sales_Data_2024_2025.csv')

df.head()

,Order ID,Order Date,Customer Name,Region,City,Category,Sub-Category,Product Name,Quantity,Unit Price,Discount,Sales,Profit,Payment Mode
0,10001,2024-10-19,Kashvi Varty,South,Bangalore,Books,Non-Fiction,Non-Fiction Ipsum,2,36294,5,68958.6,10525.09,Debit Card
1,10002,2025-08-30,Advik Desai,North,Delhi,Groceries,Rice,Rice Nemo,1,42165,20,33732.0,6299.66,Debit Card
2,10003,2023-11-04,Rhea Kalla,East,Patna,Kitchen,Juicer,Juicer Odio,4,64876,20,207603.2,19850.27,Credit Card
3,10004,2025-05-23,Anika Sen,East,Kolkata,Groceries,Oil,Oil Doloribus,5,37320,15,158610.0,36311.02,UPI
4,10005,2025-01-19,Akarsh Kaul,West,Pune,Clothing,Kids Wear,Kids Wear Quo,1,50037,10,45033.3,9050.04,Debit Card


In [3]:
df.shape

(5000, 14)

In [4]:
df.columns

Index(['Order ID', 'Order Date', 'Customer Name', 'Region', 'City', 'Category',
       'Sub-Category', 'Product Name', 'Quantity', 'Unit Price', 'Discount',
       'Sales', 'Profit', 'Payment Mode'],
      dtype='object')

In [5]:
print('Missing values Percentage: \n\n', round (df.isnull().sum().sort_values(ascending=False)/len(df)*100,1))

Missing values Percentage: 

 Order ID         0.0
Order Date       0.0
Customer Name    0.0
Region           0.0
City             0.0
Category         0.0
Sub-Category     0.0
Product Name     0.0
Quantity         0.0
Unit Price       0.0
Discount         0.0
Sales            0.0
Profit           0.0
Payment Mode     0.0
dtype: float64


# Column categories

we are going to see if the columns are categorical, or continuous. 


In [8]:

def columnCategory(dataset, categoryTotal=10):
    """
    Displays an orderly breakdown of every column. 
    Lists specific values for low cardinality columns, and flags 
    high cardinality columns without flooding the screen.
    """
    print(f"=== DATAFRAME CARDINALITY & CATEGORY ANALYSIS ===")
    print(f" Threshold for Low Cardinality: <= {categoryTotal} unique values\n")
    print("=" * 60)
    
    for column in dataset.columns:
        # Get unique values and ignore null values for an accurate count
        unique_vals = dataset[column].dropna().unique()
        unique_count = len(unique_vals)
        
        print(f"📌 Column: {column}")
        print(f"   • Total Unique Values: {unique_count}")
        
        # Condition A: Low Cardinality (Categorical)
        if unique_count <= categoryTotal:
            print(f"   • Status: [ LOW CARDINALITY / CATEGORY ]")
            # Sort the unique values for an orderly display
            try:
                sorted_vals = sorted(unique_vals)
            except TypeError:
                sorted_vals = sorted(unique_vals, key=str)
                
            print(f"   • Categories: {sorted_vals}")
            
        # Condition B: High Cardinality
        else:
            print(f"   • Status: [ HIGH CARDINALITY ]")
            print(f"   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.")
            
        print("-" * 60)

columnCategory(df, 20)

=== DATAFRAME CARDINALITY & CATEGORY ANALYSIS ===
 Threshold for Low Cardinality: <= 20 unique values

📌 Column: Order ID
   • Total Unique Values: 5000
   • Status: [ HIGH CARDINALITY ]
   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.
------------------------------------------------------------
📌 Column: Order Date
   • Total Unique Values: 730
   • Status: [ HIGH CARDINALITY ]
   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.
------------------------------------------------------------
📌 Column: Customer Name
   • Total Unique Values: 4844
   • Status: [ HIGH CARDINALITY ]
   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.
------------------------------------------------------------
📌 Column: Region
   • Total Unique Values: 4
   • Status: [ LOW CARDINALITY / CATEGORY ]
   • Categories: ['East', 'North', 'South', 'West']
------------------------------------------------------------
📌 Column: City

as we can see there are few who have low cardinality such as region, city, payment mode, profit, Payment mode, Discount, quantity, sub category and category. 

# Extract the Order Data



In [11]:
# Ensure Order Date is in the proper datetime format
df['Order Date'] = pd.to_datetime(df['Order Date'])

df['year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.strftime('%B')  # Outputs: January, February, etc.
df['Quarter'] = df['Order Date'].dt.quarter.astype(str) #Outputs: Q1, Q2, Q3, Q4

# Base Unit Cost (Operational Metric)
dataset gives you Unit Price (what the customer paid) and Profit (what the company kept), but it doesn't explicitly state how much it actually cost the company to manufacture or acquire the product.

In [14]:
# Calculate the manufacturing/acquisition cost per individual item
df['Unit Cost'] = round(df['Unit Price'] - (df['Profit'] / df['Quantity']),2)

In [15]:
df.head()

,Order ID,Order Date,Customer Name,Region,City,Category,Sub-Category,Product Name,Quantity,Unit Price,Discount,Sales,Profit,Payment Mode,year,Month,Quarter,Unit Cost
0,10001,2024-10-19,Kashvi Varty,South,Bangalore,Books,Non-Fiction,Non-Fiction Ipsum,2,36294,5,68958.6,10525.09,Debit Card,2024,October,4,31031.46
1,10002,2025-08-30,Advik Desai,North,Delhi,Groceries,Rice,Rice Nemo,1,42165,20,33732.0,6299.66,Debit Card,2025,August,3,35865.34
2,10003,2023-11-04,Rhea Kalla,East,Patna,Kitchen,Juicer,Juicer Odio,4,64876,20,207603.2,19850.27,Credit Card,2023,November,4,59913.43
3,10004,2025-05-23,Anika Sen,East,Kolkata,Groceries,Oil,Oil Doloribus,5,37320,15,158610.0,36311.02,UPI,2025,May,2,30057.80
4,10005,2025-01-19,Akarsh Kaul,West,Pune,Clothing,Kids Wear,Kids Wear Quo,1,50037,10,45033.3,9050.04,Debit Card,2025,January,1,40986.96


# RELEASE THE CSV 

In [16]:
df.to_csv('ecommerce_sales_cleaned.csv', index=False)